In [2]:
%cd /home/asehgal/formulacode/datasmith
import datetime
import json
from pathlib import Path

import pandas as pd

from datasmith.benchmark.collection import BenchmarkCollection
from datasmith.docker.context import ContextRegistry, Task
from datasmith.notebooks.utils import merge_registries, update_cr
from datasmith.utils import _get_github_metadata

curr_date: str = datetime.datetime.now().isoformat()

/home/asehgal/formulacode/datasmith


In [3]:
# update the pipeflush entries.
results_pth = Path("scratch/artifacts/pipeflush/")

registries = results_pth.rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))
registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))
# remove all entries for dask-dask and dask-distributed.``
print(len(registry.registry))
# scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T00:38:42.145419.json : 1016 entries
# registry.save_to_file(results_pth / f"merged_context_registry_{curr_date}.json")

# (optionally) Make sure that all all the registry items are in scratch/artifacts/pipeflush/commits_perfonly.parquet
# df = pd.read_parquet(results_pth / "commits_perfonly.parquet")
# registry_shas = {t.sha for t in registry.registry if t.sha is not None}
# df_shas = set(df["sha"].to_list())
# assert registry_shas.issubset(df_shas)

scratch/artifacts/pipeflush/context_registry.json : 7 entries
scratch/artifacts/pipeflush/merged_context_registry_2025-09-16T03:36:30.617775.json : 1642 entries
scratch/artifacts/pipeflush/merged_context_registry_2025-09-16T07:48:19.492393.json : 1642 entries
scratch/artifacts/pipeflush/tiny/context_registry.json : 8 entries
scratch/artifacts/pipeflush/chunk_2/context_registry.json : 1735 entries
scratch/artifacts/pipeflush/chunk_0/context_registry.json : 749 entries
scratch/artifacts/pipeflush/chunk_1/context_registry.json : 7 entries
1642


In [4]:
results_pth = Path("scratch/")

In [5]:
collections = [BenchmarkCollection.load(p) for p in (results_pth).rglob("*breakpoints.fc.pkl")]
list((results_pth).rglob("*breakpoints.fc.pkl"))

[PosixPath('scratch/artifacts/processed/downloads/pandas2/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scipy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scikit-image/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/astropy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/numpy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pandas/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pymc3/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/sklearn/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/xarray/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/dask/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/distributed/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/joblib/breakpoints.fc.pkl')]

In [ ]:
collections = [BenchmarkCollection.load(p) for p in results_pth.rglob("*breakpoints.fc.pkl")]
registries = results_pth.rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))
registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))
# remove all entries for dask-dask and dask-distributed.
print(len(registry.registry))
# scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T00:38:42.145419.json : 1016 entries
registry.save_to_file(results_pth / f"merged_context_registry_{curr_date}.json")

scratch/merged_context_registry_2025-09-06T01:31:46.096023.json : 700 entries
scratch/merged_context_registry_2025-09-04T08:32:08.486247.json : 140 entries
scratch/merged_context_registry_2025-09-16T01:21:01.034514.json : 5513 entries
scratch/merged_context_registry_2025-09-05T20:02:28.617179.json : 540 entries
scratch/merged_context_registry_2025-09-06T21:27:11.754109.json : 1129 entries
scratch/context_registry_init.json : 6 entries
scratch/merged_context_registry_2025-09-06T06:14:21.165351.json : 975 entries
scratch/merged_context_registry_2025-09-04T23:54:53.035665.json : 178 entries
scratch/merged_context_registry_2025-09-16T03:26:41.179572.json : 5513 entries
scratch/merged_context_registry_2025-09-16T01:17:55.372491.json : 5513 entries
scratch/context_registry_updated.json : 389 entries
scratch/poseidon-artifacts/pipeflush/context_registry.json : 7 entries
scratch/poseidon-artifacts/pipeflush/tiny/context_registry.json : 8 entries
scratch/poseidon-artifacts/pipeflush/chunk_2/con

03:30:42 INFO     datasmith.docker.context: Context registry saved to scratch/merged_context_registry_2025-09-16T03:26:41.179572.json


In [12]:
bps = []
for c in collections:
    df = c.enriched_breakpoints
    df["repo_name"] = f"{c.task.owner}/{c.task.repo}"
    commits = c.commits[
        ["files_changed", "sha", "date", "file_change_summary", "message", "patch", "repo_name"]
    ].rename(columns={"sha": "gt_hash"})
    frame = (
        df.groupby(["repo_name", "hash", "gt_hash"])["delta_pct"]
        .mean()
        .reset_index()
        .rename(columns={"delta_pct": "delta_pct_mean"})
    )
    merged_frame = frame.merge(commits, on=["repo_name", "gt_hash"], how="left")
    assert all(merged_frame.notnull().all())
    bps.append(merged_frame)

all_enriched = pd.concat(bps, ignore_index=True)
useful_enriched = all_enriched[(-1 * all_enriched["delta_pct_mean"]) > 1]
print(f"Found {len(useful_enriched)} tasks with >1% mean improvement in at least one commit")
useful_enriched["repo_name"].value_counts().reset_index()

Found 362 tasks with >1% mean improvement in at least one commit


,repo_name,count
0,pandas-dev/pandas,124
1,scipy/scipy,105
2,astropy/astropy,32
3,dask/distributed,26
4,numpy/numpy,26
5,dask/dask,17
6,scikit-learn/scikit-learn,10
7,pymc-devs/pymc3,10
8,pydata/xarray,5
9,joblib/joblib,5


In [13]:
# make a csv containing:
# container_name - task.with_env("pkg").get_image_name()
# patch - git patch of the gt_sha
# message - task.get_commit_message()
# task_id - "{owner}_{repo}_{i unique to each repo}" like astropy_astropy_id1
# gt_sha - sha value of the git batch
# sha - sha value of the parent commit.
# rows = []


def get_patch(row: pd.Series) -> str | None:
    owner, repo = row["repo_name"].split("/")
    sha = row["gt_hash"]
    base_commit = row["hash"]
    # How can I change the next line to get the patch between base_commit and sha?
    # endpoint = f"/repos/{owner}/{repo}/commits/{sha}"
    endpoint = f"/repos/{owner}/{repo}/compare/{base_commit}...{sha}"
    # Ask for the commit as a unified diff (not JSON)
    diff_text = _get_github_metadata(endpoint=endpoint, params={"diff_api": "true"})
    if not diff_text or "diff" not in diff_text:
        print("No diff found")
        return None
    return diff_text["diff"]


def make_task(row) -> str:
    owner, repo = row["repo_name"].split("/")
    sha = row["hash"]
    commit_date = row["date"]
    return Task(owner=owner, repo=repo, sha=sha, commit_date=commit_date).with_tag("pkg").get_image_name()


useful_enriched["container_name"] = useful_enriched.apply(make_task, axis=1)
useful_enriched["patch"] = useful_enriched.apply(get_patch, axis=1)
useful_enriched["message"] = useful_enriched["message"]
print(f"After adding container names, {len(useful_enriched)} tasks remain")
useful_enriched = useful_enriched.dropna(subset=["patch"])
# [
#     useful_enriched["container_name"].isin({t.get_image_name() for t in registry.registry})
# ].dropna(subset=["patch"])
print(f"After filtering for available containers, {len(useful_enriched)} tasks remain")
repo_counters = {}


def compute_task_id(row):
    repo = row["repo_name"].replace("/", "_")
    if repo not in repo_counters:
        repo_counters[repo] = 0
    repo_counters[repo] += 1
    return f"{repo}_{repo_counters[repo]}"


useful_enriched["task_id"] = useful_enriched.apply(compute_task_id, axis=1)
useful_enriched["base_commit"] = useful_enriched["hash"]

out_pth = results_pth / f"useful_enriched.tbformat_{curr_date}.parquet"

useful_enriched[["container_name", "patch", "message", "task_id", "gt_hash", "base_commit", "date"]].to_parquet(
    out_pth, index=False
)
print(f"Wrote to {out_pth}")

/tmp/ipykernel_2545630/3467736609.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  useful_enriched["container_name"] = useful_enriched.apply(make_task, axis=1)


No diff found
No diff found
No diff found
No diff found
No diff found
No diff found
No diff found
After adding container names, 362 tasks remain
After filtering for available containers, 355 tasks remain


/tmp/ipykernel_2545630/3467736609.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  useful_enriched["patch"] = useful_enriched.apply(get_patch, axis=1)
/tmp/ipykernel_2545630/3467736609.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  useful_enriched["message"] = useful_enriched["message"]


Wrote to scratch/useful_enriched.tbformat_2025-09-16T03:26:41.179572.parquet


In [17]:
rows = []
for _, row in useful_enriched.iterrows():
    sha = row["base_commit"]
    repo_name = row["repo_name"]
    date = row["date"]

    rows.append({
        "container_name": row["container_name"],
        "sha": sha,
        "repo_name": repo_name,
        "date": date,
        "kind": "commit",
        "has_asv": True,
    })

df = pd.DataFrame(rows)
out_path = results_pth / f"synthetic_commits_perfonly_usefulonly_{curr_date}.parquet"
df.to_parquet(out_path, index=False)
print(f"Wrote to {out_path.resolve()}")

Wrote to /home/asehgal/formulacode/datasmith/scratch/synthetic_commits_perfonly_usefulonly_2025-09-16T03:26:41.179572.parquet


In [18]:
# all the paths relative to the current directory
# cr path
# artifacts_pth = Path("scratch/artifacts/processed")
cr_path = results_pth / f"merged_context_registry_{curr_date}.json"
enriched_path = results_pth / f"useful_enriched.tbformat_{curr_date}.parquet"
synthetic_path = results_pth / f"synthetic_commits_perfonly_usefulonly_{curr_date}.parquet"
assert cr_path.exists()
assert enriched_path.exists()
assert synthetic_path.exists()
print("cr_path\t:", cr_path)
print("enriched_path\t:", enriched_path)
print("synthetic_path\t:", synthetic_path)

cr_path	: scratch/merged_context_registry_2025-09-16T03:26:41.179572.json
enriched_path	: scratch/useful_enriched.tbformat_2025-09-16T03:26:41.179572.parquet
synthetic_path	: scratch/synthetic_commits_perfonly_usefulonly_2025-09-16T03:26:41.179572.parquet


In [8]:
cr = ContextRegistry.load_from_file(
    Path("scratch/artifacts/processed/downloads/merged_context_registry_2025-09-12T10:50:49.251932.json")
)
tasks = {t.get_image_name(): t for t in cr.registry}

df[~df["container_name"].isin(tasks)].shape

(150, 6)

In [ ]:
all_states = {}
for _, row in df.iterrows():
    repo_name = row["repo_name"]
    sha = row["sha"]
    has_asv = row.get("has_asv", True)
    if not has_asv:
        continue
    owner, repo = repo_name.split("/")
    commit_date_unix: float = (
        0.0 if row.get("date", None) is None else datetime.datetime.fromisoformat(row["date"]).timestamp()
    )
    if (owner, repo) not in all_states:
        all_states[(owner, repo)] = [(sha, commit_date_unix)]
    else:
        all_states[(owner, repo)].append((sha, commit_date_unix))

NameError: name 'dfdedup' is not defined

In [ ]:
# all_imgs = {t.get_image_name() for t in cr.registry}
tasks = []
for (owner, repo), uniq in all_states.items():
    for sha, date in list(uniq):
        task = Task(owner, repo, sha, commit_date=float(date))
        if task in cr:
            tasks.append(task.with_tag("pkg"))
        else:
            print(f"main: skipping {task} not in context registry")

Commit dates differ: self=1598788394.0, value=1597051613.0
main: skipping Task(owner='numpy', repo='numpy', sha='00a45b4dca164105b50ba29e1735e96b573b639c', commit_date=1598788394.0, tag='pkg') not in context registry
Commit dates differ: self=1594484456.0, value=1594249986.0
main: skipping Task(owner='numpy', repo='numpy', sha='1405a30b1f1100f88c38731a9170f889002d316a', commit_date=1594484456.0, tag='pkg') not in context registry
Commit dates differ: self=1617125506.0, value=1615096103.0
main: skipping Task(owner='numpy', repo='numpy', sha='37ce99a4ab6f066f1363c33d1ec6f2b4c6c4a583', commit_date=1617125506.0, tag='pkg') not in context registry
main: skipping Task(owner='numpy', repo='numpy', sha='3b3dbdda2de4d6193dd4ab299067f20fb47515f2', commit_date=1622035812.0, tag='pkg') not in context registry
main: skipping Task(owner='numpy', repo='numpy', sha='3c91a3e1704c8aa7f1258aa30892040df9d952f4', commit_date=1621868520.0, tag='pkg') not in context registry
Commit dates differ: self=1594249